# Day 4 control: reputation-modulated consensus on **IID** SST-2 — and IID vs non-IID

Reads `outputs/day5_reputation_iid.json` (IID run) and, if present,
`outputs/day5_reputation.json` (the non-IID Dirichlet run from Day 4) for a
head-to-head comparison.

**Why this run exists.** On non-IID Dirichlet(α=0.5) the E3 working window for
`β` did not transfer (see `теория/swarm-mezo.md` §4.6). The hypothesis: under
non-IID the probe loss `L_i` measures shard–probe match, not optimization
fitness, so the reputation theory's premise is violated. The IID split restores
a single shared objective across agents, making `L_i` a clean fitness signal
again — the regime where the §4 reputation theory is actually defined.

**The question.** Does a working `β` window appear under IID while staying
absent under non-IID? If yes, the §4.6 diagnosis is confirmed: the idea of
reputational selection is sound, but its fitness signal is corrupted by
non-IID data heterogeneity.

Same setup as Day 4 otherwise: N=8, K=100, 5000 steps, `W_ij = r_j/Σr`,
`r_i ← r_i/(γ_r + β·|L_i − L_min|)`, sweep `β ∈ {0, 0.1, 0.5, 1, 10}`.

Five views:
1. Val accuracy & loss vs step — IID, β sweep.
2. **Headline** — final val accuracy vs β, IID vs non-IID side by side.
3. Reputation concentration over rounds — IID cascade detector.
4. Per-agent probe loss at the final round — IID.
5. Summary table — IID vs non-IID.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


def load_sweep(fname):
    """Load a reputation sweep JSON; return None if the run hasn't happened yet."""
    p = ROOT / 'outputs' / fname
    if not p.exists():
        return None
    d = json.loads(p.read_text())
    # Sort runs by β so curves and bars stack predictably.
    d['runs'] = dict(sorted(d['runs'].items(), key=lambda kv: kv[1]['beta']))
    return d


iid    = load_sweep('day5_reputation_iid.json')
noniid = load_sweep('day5_reputation.json')

if iid is None:
    print('outputs/day5_reputation_iid.json не найден.')
    print('Запустите IID-прогон: notebooks/colab_run_reputation_iid.ipynb')
    print('или локально:  SHARDING=iid uv run python scripts/run_reputation.py')
else:
    print('IID config :', iid['config'])
    print('IID runs   :', list(iid['runs'].keys()))
if noniid is not None:
    print('non-IID runs:', list(noniid['runs'].keys()))
else:
    print('non-IID JSON отсутствует — сравнительные графики покажут только IID.')

cmap = plt.get_cmap('viridis')
plt.rcParams['figure.dpi'] = 110

## Plot 1 — val accuracy & loss vs step (IID)

In [ ]:
assert iid is not None, 'сначала прогон IID-свипа'
runs = iid['runs']
colors = {k: cmap(i / max(len(runs) - 1, 1)) for i, k in enumerate(runs)}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, key, ylabel in [
    (axes[0], 'eval_acc',  'SST-2 val accuracy'),
    (axes[1], 'eval_loss', 'SST-2 val CE loss'),
]:
    for name, h in runs.items():
        ax.plot(h['eval_step'], h[key], marker='o', markersize=3,
                color=colors[name], label=f"β={h['beta']}")
    ax.set_xlabel('per-agent step'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} — β sweep (IID)')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle(
    f"Day 4 control: reputational MeZO on IID SST-2 "
    f"(N={iid['config']['n_agents']}, K={iid['config']['local_steps']})",
    y=1.02,
)
plt.tight_layout()

## Plot 2 — final val accuracy vs β: IID vs non-IID (the headline)

If reputational selection has a working window, it should show here as IID
bars rising above their β=0 baseline for small β. The non-IID bars (Day 4)
monotonically fall — that is the §4.6 result. The contrast between the two
groups is the whole experiment.

In [ ]:
assert iid is not None, 'сначала прогон IID-свипа'
betas = [h['beta'] for h in iid['runs'].values()]
final_iid = [h['eval_acc'][-1] for h in iid['runs'].values()]
x = np.arange(len(betas))

fig, ax = plt.subplots(figsize=(8.5, 5))

if noniid is not None:
    w = 0.38
    nmap = {h['beta']: h['eval_acc'][-1] for h in noniid['runs'].values()}
    final_non = [nmap.get(b, np.nan) for b in betas]
    b1 = ax.bar(x - w / 2, final_iid, w, color='tab:green', label='IID')
    b2 = ax.bar(x + w / 2, final_non, w, color='tab:gray',
                label='non-IID (Dirichlet α=0.5)')
    bars_for_labels = list(b1) + list(b2)
    vals_for_labels = final_iid + final_non
else:
    w = 0.6
    b1 = ax.bar(x, final_iid, w, color='tab:green', label='IID')
    bars_for_labels, vals_for_labels = list(b1), final_iid

for bar, val in zip(bars_for_labels, vals_for_labels):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002,
                f'{val:.3f}', ha='center', fontsize=8)

# β=0 IID baseline — reputation is off there by construction (all r_i equal).
ax.axhline(final_iid[0], color='tab:green', linestyle='--', alpha=0.5,
           label=f'IID β=0 baseline = {final_iid[0]:.3f}')
ax.set_xticks(x); ax.set_xticklabels([str(b) for b in betas])
ax.set_xlabel('β'); ax.set_ylabel('final SST-2 val accuracy')
ax.set_title('Final accuracy vs β — does a working window appear under IID?')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

## Plot 3 — reputation concentration over rounds (IID)

`max_i r_i / Σ_j r_j` per consensus round. Uniform = `1/N = 0.125`. Under IID
the loss spread between agents is pure SPSA noise rather than shard skew, so
the concentration should stay closer to uniform for moderate β than it did
on non-IID.

In [ ]:
assert iid is not None, 'сначала прогон IID-свипа'
N = iid['config']['n_agents']
fig, ax = plt.subplots(figsize=(8, 4.5))

for name, h in iid['runs'].items():
    reps = np.asarray(h['reputations'])
    if reps.size == 0:
        continue
    shares = reps / reps.sum(axis=1, keepdims=True)
    max_share = shares.max(axis=1)
    ax.plot(np.arange(1, len(max_share) + 1), max_share, marker='o',
            markersize=3, color=colors[name], label=f"β={h['beta']}")

ax.axhline(1 / N, color='gray', linestyle=':', alpha=0.6,
           label=f'uniform = 1/N = {1/N:.3f}')
ax.set_xlabel('consensus round')
ax.set_ylabel('max reputation share')
ax.set_title('Reputation concentration over time — IID (cascade detector)')
ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()

## Plot 4 — per-agent probe loss at the final round (IID)

Under IID all agents share one distribution, so a tight loss spread is
expected — and any agent that still "led" did so on noise, not on an easier
shard.

In [ ]:
assert iid is not None, 'сначала прогон IID-свипа'
fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(N)
width = 0.8 / max(len(iid['runs']), 1)
for i, (name, h) in enumerate(iid['runs'].items()):
    losses = h.get('consensus_eval_losses') or []
    if not losses:
        continue
    final = np.asarray(losses[-1])
    ax.bar(x + i * width - 0.4 + width / 2, final, width,
           color=colors[name], label=f"β={h['beta']}")

ax.set_xticks(x)
ax.set_xticklabels([f'agent {i}' for i in x], rotation=30, ha='right')
ax.set_ylabel('probe-batch CE loss (final round)')
ax.set_title('Per-agent loss at end of training — IID')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()

## Summary table — IID vs non-IID

In [ ]:
assert iid is not None, 'сначала прогон IID-свипа'
nmap = ({h['beta']: h for h in noniid['runs'].values()}
        if noniid is not None else {})


def max_rep_share(h):
    reps = np.asarray(h['reputations']) if h['reputations'] else np.zeros((0, N))
    if reps.size == 0:
        return float('nan')
    return float((reps[-1] / reps[-1].sum()).max())


rows = []
for h in iid['runs'].values():
    b = h['beta']
    row = {
        'β':                 b,
        'IID val_acc':       h['eval_acc'][-1],
        'IID max rep share': max_rep_share(h),
    }
    if b in nmap:
        row['non-IID val_acc']       = nmap[b]['eval_acc'][-1]
        row['non-IID max rep share'] = max_rep_share(nmap[b])
        row['IID − non-IID acc']     = h['eval_acc'][-1] - nmap[b]['eval_acc'][-1]
    rows.append(row)

pd.DataFrame(rows).round(4)

## Чтение результата

- **Окно `β` появилось под IID, отсутствует под non-IID** → диагноз §4.6
  подтверждён: отбор сам по себе работоспособен, его ломает именно
  non-IID-искажение сигнала фитнеса `L_i`. Вывод для теории: репутации нужен
  сигнал, развязанный от трудности шарда.
- **Окна нет даже под IID** → проблема глубже постановки данных
  (мультипликативная динамика репутаций, шум 32-примерного probe). Тогда под
  вопросом сам механизм `r_i ← r_i/(γ_r+β·|L_i−L_min|)`, а не только non-IID.

В обоих случаях результат честный и уточняет постановку — см. §4.6 и §7 в
`теория/swarm-mezo.md`.